# LangGraph RAG 테스트 노트북

이 노트북은 LangGraph 기반 Routing RAG 시스템을 테스트합니다.

## 워크플로우

```
START → route → [조건부 분기]
                ├─ search → retrieve → generate → END
                └─ direct → generate → END
```

1. **Route**: 질문 분석 → search/direct 판단
2. **Retrieve**: Hybrid Search로 top-3 검색 (조건부)
3. **Generate**: 답변 생성

## 1. 환경 설정 확인

In [ ]:
import sys
import os

# 현재 디렉토리 확인
print(f"Current directory: {os.getcwd()}")

# Python 버전 확인
print(f"Python version: {sys.version}")

## 2. 필수 패키지 Import

In [ ]:
# Import 테스트
try:
    import langgraph
    print("✓ langgraph imported")
except ImportError as e:
    print(f"✗ langgraph import failed: {e}")

try:
    from langchain_openai import ChatOpenAI
    print("✓ langchain_openai imported")
except ImportError as e:
    print(f"✗ langchain_openai import failed: {e}")

try:
    from langgraph_rag import LangGraphRAG
    print("✓ langgraph_rag imported")
except ImportError as e:
    print(f"✗ langgraph_rag import failed: {e}")
    print(f"\nError details: {e}")
    print("\nMake sure:")
    print("1. .env file exists with OPENAI_API_KEY")
    print("2. Database is running and initialized")
    print("3. All dependencies are installed")

## 3. 환경 변수 확인

In [ ]:
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경 변수 확인
api_key = os.getenv("OPENAI_API_KEY")
db_url = os.getenv("DATABASE_URL")

print(f"OPENAI_API_KEY: {'✓ Set' if api_key else '✗ Not set'}")
print(f"DATABASE_URL: {'✓ Set' if db_url else '✗ Not set'}")

if db_url:
    # 비밀번호는 숨기고 출력
    safe_url = db_url.split('@')[0].split(':')[0:2]
    print(f"Database: {safe_url[0]}://...@{db_url.split('@')[1] if '@' in db_url else 'localhost'}")

## 4. 데이터베이스 연결 테스트

In [ ]:
from search_app.database import Database
from search_app.config import Config

# 데이터베이스 연결 테스트
try:
    db = Database()
    db.connect()
    
    # 테이블 존재 확인
    db.execute(f"""
        SELECT COUNT(*) FROM {Config.TABLE_NAME}
    """)
    count = db.cur.fetchone()[0]
    
    print(f"✓ Database connected")
    print(f"✓ Table '{Config.TABLE_NAME}' exists")
    print(f"✓ Total products: {count}")
    
    db.close()
except Exception as e:
    print(f"✗ Database connection failed: {e}")
    print("\nMake sure:")
    print("1. ParadeDB Docker is running")
    print("2. Database is initialized (python -m search_app.setup)")

## 5. LangGraph RAG 초기화

In [ ]:
# LangGraph RAG 인스턴스 생성 (디버그 모드)
rag = LangGraphRAG(debug=True)

print("✓ LangGraph RAG initialized")
print(f"✓ LLM model: gpt-5-mini")
print(f"✓ Database connected")
print(f"✓ Hybrid search ready")

## 6. 테스트 1: Direct Question (검색 불필요)

간단한 인사나 일반적인 질문은 검색 없이 바로 답변합니다.

In [ ]:
question1 = "안녕하세요"

print(f"\n{'='*80}")
print(f"질문: {question1}")
print(f"{'='*80}\n")

answer1 = rag.run(question1)

print(f"\n{'='*80}")
print(f"답변:\n{answer1}")
print(f"{'='*80}")

## 7. 테스트 2: Search Question (검색 필요)

특정 대출 상품을 찾는 질문은 Hybrid Search를 수행합니다.

In [ ]:
question2 = "의사 전용 대출 상품 추천해줘"

print(f"\n{'='*80}")
print(f"질문: {question2}")
print(f"{'='*80}\n")

answer2 = rag.run(question2)

print(f"\n{'='*80}")
print(f"답변:\n{answer2}")
print(f"{'='*80}")

## 8. 테스트 3: Low Interest Rate Search

저금리 대출을 찾는 구체적인 검색 질문입니다.

In [ ]:
question3 = "저금리 대출 상품을 찾고 있어요"

print(f"\n{'='*80}")
print(f"질문: {question3}")
print(f"{'='*80}\n")

answer3 = rag.run(question3)

print(f"\n{'='*80}")
print(f"답변:\n{answer3}")
print(f"{'='*80}")

## 9. 테스트 4: 직접 질문해보기

원하는 질문을 입력하고 테스트해보세요.

In [ ]:
# 여기에 원하는 질문을 입력하세요
custom_question = "청년 대상 대출 상품이 있나요?"

print(f"\n{'='*80}")
print(f"질문: {custom_question}")
print(f"{'='*80}\n")

custom_answer = rag.run(custom_question)

print(f"\n{'='*80}")
print(f"답변:\n{custom_answer}")
print(f"{'='*80}")

## 10. 디버그 모드 OFF로 테스트

디버그 메시지 없이 깔끔한 결과만 확인합니다.

In [ ]:
# 기존 RAG 인스턴스 종료
rag.close()

# 디버그 모드 OFF로 새로 생성
rag_clean = LangGraphRAG(debug=False)

print("✓ Clean mode RAG initialized\n")

In [ ]:
# 깔끔한 모드로 질문
clean_question = "주택담보대출 상품 추천해주세요"

print(f"질문: {clean_question}\n")
clean_answer = rag_clean.run(clean_question)
print(f"답변:\n{clean_answer}")

## 11. 여러 질문 배치 테스트

In [ ]:
# 다양한 질문 목록
test_questions = [
    "감사합니다",  # Direct
    "전세자금대출 상품 알려주세요",  # Search
    "금리가 낮은 순서로 대출 상품 추천해주세요",  # Search
    "중소기업 대출 상품이 있나요?",  # Search
]

results = []

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*80}")
    print(f"[{i}/{len(test_questions)}] {q}")
    print(f"{'='*80}")
    
    answer = rag_clean.run(q)
    results.append({"question": q, "answer": answer})
    
    print(f"\n답변: {answer[:200]}..." if len(answer) > 200 else f"\n답변: {answer}")

print(f"\n\n{'='*80}")
print(f"✓ All {len(test_questions)} tests completed")
print(f"{'='*80}")

## 12. 결과 요약

In [ ]:
import pandas as pd

# 결과를 DataFrame으로 표시
df = pd.DataFrame(results)
df.index = df.index + 1
df.index.name = "No."

# 답변이 길면 줄여서 표시
df["answer_preview"] = df["answer"].apply(lambda x: x[:100] + "..." if len(x) > 100 else x)

print("\n테스트 결과 요약:")
print(df[["question", "answer_preview"]])

## 13. 리소스 정리

In [ ]:
# RAG 인스턴스 종료
rag_clean.close()

print("✓ Resources cleaned up")

## 14. 추가 실험: State 직접 확인

LangGraph의 State를 직접 확인해볼 수 있습니다.

In [ ]:
# 새 RAG 인스턴스 생성
rag_debug = LangGraphRAG(debug=False)

# Initial state 생성
from langgraph_rag import RAGState

initial_state: RAGState = {
    "question": "의사 전용 대출",
    "route_decision": "",
    "search_results": [],
    "answer": "",
    "debug": False
}

# Graph 실행 및 state 추적
print("Initial State:")
print(f"  question: {initial_state['question']}")
print(f"  route_decision: {initial_state['route_decision']}")
print(f"  search_results: {len(initial_state['search_results'])} items")
print(f"  answer: {initial_state['answer'][:50] if initial_state['answer'] else 'empty'}")

# Graph 실행
final_state = rag_debug.graph.invoke(initial_state)

print("\nFinal State:")
print(f"  question: {final_state['question']}")
print(f"  route_decision: {final_state['route_decision']}")
print(f"  search_results: {len(final_state['search_results'])} items")
print(f"  answer: {final_state['answer'][:100]}...")

if final_state['search_results']:
    print("\nTop 3 Search Results:")
    for i, result in enumerate(final_state['search_results'][:3], 1):
        print(f"  {i}. {result['product_name']} (score: {result['rrf_score']:.4f})")

rag_debug.close()

## 정리

이 노트북에서 다음을 테스트했습니다:

1. ✓ 환경 설정 확인
2. ✓ 패키지 Import
3. ✓ 데이터베이스 연결
4. ✓ Direct Question (검색 불필요)
5. ✓ Search Question (Hybrid Search)
6. ✓ 다양한 질문 유형
7. ✓ 디버그 모드 / 클린 모드
8. ✓ 배치 테스트
9. ✓ State 추적

### 다음 단계

- 더 많은 질문으로 테스트
- 라우팅 로직 튜닝
- 검색 결과 개수 조정
- 프롬프트 개선